In [1]:
import pandas as pd
import datetime

In [2]:
series_id = '2149' # Id number from the matchplay series

In [3]:
# Parameters
series_id = "2456"


In [4]:
#papermill_description=load_standings_csv
standings_df = pd.read_csv('mfp_league_'+series_id+'_standings.csv', encoding='utf-8')
standings_df.set_index('player_id',inplace=True)

In [5]:
top_scores_displayed = ['top_score_1','top_score_2','top_score_3','top_score_4','top_score_5','top_score_6']
if (int(series_id) < 1875):
    top_scores_displayed = top_scores_displayed[:5]
if standings_df.weeks_played.max() < 6:
    top_scores_displayed = top_scores_displayed[:standings_df.weeks_played.max()]


In [6]:
#papermill_description=load_matches_csv
matches = pd.read_csv('mfp_league_'+series_id+'_full.csv')

In [7]:
matches['7pts'] = matches.points == 7
matches['5pts'] = matches.points == 5
matches['4pts'] = matches.points == 4
matches['3pts'] = matches.points == 3
matches['1pts'] = matches.points == 1

In [8]:
player_wins = matches.groupby('players')[['7pts','5pts','4pts','3pts','1pts']].sum()

In [9]:
display_cols = ['current_score']+top_scores_displayed+['weeks_played','weeks_won','avg_score','proj_score','cur_total','7pts','5pts','4pts','3pts','1pts',
 'max_score_possible','required_avg_win','required_avg_trophy']

In [10]:
#papermill_description=adjusting_columns_displayed
if standings_df.weeks_played.max() == 10:
    display_cols.remove('max_score_possible')
    display_cols.remove('required_avg_win')
    display_cols.remove('required_avg_trophy')
    display_cols.remove('proj_score')
    #standings_df = standings_df[standings_df.weeks_played >= 5]

In [11]:
standings_web_df = pd.merge(standings_df,player_wins,how='left',left_index = True,right_index=True)

In [12]:
standings_web_df = standings_web_df[display_cols].reset_index()
standings_web_df.sort_values('current_score',ascending=False,inplace=True)

In [13]:
# def make_pretty(styler):
#     styler.set_caption("Last Updated: ")
#     styler.format(rain_condition)
#     styler.format_index(lambda v: v.strftime("%A"))
#     styler.background_gradient(axis=None, vmin=1, vmax=5, cmap="YlGnBu")
#     return styler

In [14]:
standings_web_df.to_csv('mfp_league_web_'+series_id+'.csv',index=False)

In [15]:
#papermill_description=writing_to_html

html_string = '''
<html>
  <head><title>{title}</title></head>
  <link rel="stylesheet" type="text/css" href="mfp_style.css"/>
  <body>
    {table}
    Last Updated: {time}
  </body>
  <script src="sortable.js"></script>
</html>
'''

with open('mfp_league_'+series_id+'_standings.html', 'w') as f:
    f.write(html_string.format(title="MFP League "+series_id+" standings",
                               table=standings_web_df.to_html(classes='mfp_style',sparsify=True,index=False),time=datetime.datetime.now()))

In [16]:
import pickle

with open('ftp-creds.pickle', 'rb') as f:
    creds = pickle.load(f)

In [17]:
#papermill_description=uploading_to_ftp
import ftplib
session = ftplib.FTP(creds['host'],creds['user'],creds['password'])
file = open('mfp_league_'+series_id+'_standings.html','rb')                  # file to send
session.cwd('public_html/mfpinball.org/stats')
session.storbinary('STOR mfp_league_'+series_id+'_standings.html', file)     # send the file
file.close()                                    # close file and FTP
session.quit()

'221-Goodbye. You uploaded 25 and downloaded 0 kbytes.\n221 Logout.'